# Part 1: Genesis Franka Pick-and-Place

This notebook runs the phase-1 path from `PLAN.md`: headless Genesis, IK expert rollout, LeRobot dataset export, reload, and rollout video display.

In [1]:
from pathlib import Path

from IPython.display import Video, display

from env import TinyPickPlaceConfig
from part1_simulator import (
    LeRobotPickPlaceDataset,
    collect_genesis_franka_lerobot_dataset,
    make_pick_place_dataloader,
)

dataset_root = Path("datasets/simple-vla-genesis-franka-pick-place")
video_dir = Path("rollouts/part1")

config = TinyPickPlaceConfig(n_episodes=1, image_size=96, seed=0)

In [2]:
collect_genesis_franka_lerobot_dataset(
    root=dataset_root,
    config=config,
    n_episodes=1,
    steps_per_segment=8,
    image_size=96,
    show_viewer=False,
    backend="gpu",
    video_dir=video_dir,
)

/home/haguruma/SourceCode/simple-vla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 06/01/26 04:42:39.628 398627] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[Genesis] [04:42:41] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [04:42:41] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [04:42:41] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [04:42:42] [INFO] Running on [NVIDIA GeForce RTX 3090] with backend gs.cuda. Device memory: 24.00 GB.
[Genesis] [04:42:42] [INFO] 🚀 Genesis initialized. 🔖 version: 1.0.0, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [04:42:44] [INFO] Scene <d5db5ba> created.
[Genesis] [04:42:44] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 0, uid: <17e6dcc>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [04:42:44] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 1, uid: <e6ea326>, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [04:42:44] [INFO] Adding <gs.engine.entities.RigidEntity>. idx: 2, uid: <23db7c0>, morph: <gs.morphs.MJCF(file='/home/hagu

Map: 100%|██████████| 48/48 [00:00<00:00, 1265.46 examples/s]


LeRobotDataset({
    Repository ID: 'local/simple-vla-genesis-franka-pick-place',
    Number of selected episodes: '1',
    Number of selected samples: '48',
    Features: '['observation.images.camera', 'observation.robot_qpos', 'observation.state', 'action', 'object_pose', 'target_pose', 'object_xyz', 'target_xyz', 'gripper_width', 'label', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})

In [3]:
dataset = LeRobotPickPlaceDataset(dataset_root, chunk_size=8)
sample = dataset[0]

print("frames:", len(dataset))
print("image:", tuple(sample["image"].shape))
print("state:", tuple(sample["state"].shape))
print("action:", tuple(sample["action"].shape))
print("action_chunk:", tuple(sample["action_chunk"].shape))
print("task:", sample["task"])

frames: 48
image: (3, 96, 96)
state: (16,)
action: (9,)
action_chunk: (8, 9)
task: pick the blue cube and place it on the target


In [4]:
loader = make_pick_place_dataloader(dataset, batch_size=4, shuffle=False)
batch = next(iter(loader))
print(batch["image"].shape, batch["state"].shape, batch["action"].shape, batch["action_chunk"].shape)

torch.Size([4, 3, 96, 96]) torch.Size([4, 16]) torch.Size([4, 9]) torch.Size([4, 8, 9])


In [5]:
display(Video(str(video_dir / "episode_000.mp4"), embed=True))